Amardeep Singh

E23CSEU2189

In [1]:
import torch
import torch.nn as nn
import numpy as np

# -------------------------
# Dataset
# -------------------------
text = """machine learning models learn patterns from data
sequence models process data step by step
recurrent neural networks are designed for sequential tasks
rnn models maintain hidden states across time steps
long short term memory networks solve long dependency problems
lstm uses gates to control information flow
gru models simplify the lstm architecture
sequence prediction is useful in many applications
language modeling predicts the next word in a sentence
speech recognition processes audio sequences
time series forecasting predicts future values
music generation creates new melodies
generative models learn probability distributions
they generate new samples similar to training data
sequence generation is widely used in artificial intelligence
deep learning improves sequence modeling performance"""

# -------------------------
# Preprocessing
# -------------------------
words = text.split()
vocab = list(set(words))
word_to_ix = {word: i for i, word in enumerate(vocab)}
ix_to_word = {i: word for word, i in word_to_ix.items()}

# Create sequences
seq_length = 4
data = []
for i in range(len(words) - seq_length):
    seq = words[i:i+seq_length]
    target = words[i+seq_length]
    data.append((seq, target))

# -------------------------
# Model
# -------------------------
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_size=64, hidden_size=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

# -------------------------
# Training
# -------------------------
model = LSTMModel(len(vocab))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    total_loss = 0
    for seq, target in data:
        seq_idx = torch.tensor([word_to_ix[w] for w in seq]).unsqueeze(0)
        target_idx = torch.tensor([word_to_ix[target]])

        output = model(seq_idx)
        loss = loss_fn(output, target_idx)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

# -------------------------
# Generate Sequence
# -------------------------
def generate(seed, length=10):
    model.eval()
    words_gen = seed.copy()

    for _ in range(length):
        seq_idx = torch.tensor([word_to_ix[w] for w in words_gen[-seq_length:]]).unsqueeze(0)
        with torch.no_grad():
            output = model(seq_idx)
        predicted = torch.argmax(output).item()
        words_gen.append(ix_to_word[predicted])

    return " ".join(words_gen)

# Example
seed = ["machine", "learning", "models", "learn"]
print("Generated Text:")
print(generate(seed))

Epoch 0, Loss: 499.4173
Epoch 20, Loss: 0.1265
Epoch 40, Loss: 0.0278
Epoch 60, Loss: 0.0081
Epoch 80, Loss: 0.0025
Generated Text:
machine learning models learn patterns from data sequence models process data step by step


In [2]:
import torch
import torch.nn as nn

# Reuse same dataset
words = text.split()
vocab = list(set(words))
word_to_ix = {word: i for i, word in enumerate(vocab)}
ix_to_word = {i: word for word, i in word_to_ix.items()}

seq_length = 4

data = []
for i in range(len(words) - seq_length):
    seq = words[i:i+seq_length]
    target = words[i+seq_length]
    data.append((seq, target))

# -------------------------
# Transformer Model
# -------------------------
class TransformerModel(nn.Module):
    def __init__(self, vocab_size, embed_size=64, num_heads=2, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.pos_encoder = nn.Parameter(torch.zeros(1, seq_length, embed_size))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_size, nhead=num_heads
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.fc = nn.Linear(embed_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x) + self.pos_encoder
        x = x.permute(1, 0, 2)  # seq_len, batch, embed
        out = self.transformer(x)
        out = self.fc(out[-1])
        return out

# -------------------------
# Training
# -------------------------
model = TransformerModel(len(vocab))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    total_loss = 0
    for seq, target in data:
        seq_idx = torch.tensor([word_to_ix[w] for w in seq]).unsqueeze(0)
        target_idx = torch.tensor([word_to_ix[target]])

        output = model(seq_idx)
        loss = loss_fn(output, target_idx)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

# -------------------------
# Generate Sequence
# -------------------------
def generate_transformer(seed, length=10):
    model.eval()
    words_gen = seed.copy()

    for _ in range(length):
        seq_idx = torch.tensor([word_to_ix[w] for w in words_gen[-seq_length:]]).unsqueeze(0)
        with torch.no_grad():
            output = model(seq_idx)
        predicted = torch.argmax(output).item()
        words_gen.append(ix_to_word[predicted])

    return " ".join(words_gen)

# Example
seed = ["sequence", "models", "process", "data"]
print("Generated Text (Transformer):")
print(generate_transformer(seed))

/tmp/ipykernel_2335/2009452028.py:30: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


Epoch 0, Loss: 571.0217
Epoch 20, Loss: 485.1497
Epoch 40, Loss: 487.1391
Epoch 60, Loss: 477.5906
Epoch 80, Loss: 480.8378
Generated Text (Transformer):
sequence models process data sequence sequence sequence sequence sequence sequence sequence sequence sequence sequence
